<a href="https://colab.research.google.com/github/emmanuelmassawe/-Lloyds-Bank-Group-Customer-Churn-Prediction/blob/main/Sentiment_Analysis_with_huggingface_transformers_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U transformers
!pip install -U kagglehub
!pip install -U kaggle
!pip install -U evaluate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 96.4 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.21.0
    Uninstalling transformers-5.21.0:
      Successfully uninstalled transformers-5.21.0
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.0.2
    Uninstalling kaggle-2.0.2:
      Successfully uninstalled kaggle-2.0.2


## importing necessary file

In [3]:
#importing necessary libraries
import pandas as pd
import argparse

import numpy as np
import evaluate
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kashishparmar02/social-media-sentiments-analysis-dataset")

print("Path to dataset files:", path)


100%|██████████| 50.9k/50.9k [00:00<00:00, 49.0MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/kashishparmar02/social-media-sentiments-analysis-dataset/versions/3


In [5]:
import os

print(os.listdir(path))

['sentimentdataset.csv']


## loading the dataset

In [78]:
df =pd.read_csv(os.path.join(path,'sentimentdataset.csv'))

In [79]:
df.head(10)

,Unnamed: 0.1,Unnamed: 0,Text,Sentiment,Timestamp,User,Platform,Hashtags,Retweets,Likes,Country,Year,Month,Day,Hour
0,0,0,Enjoying a beautiful day at the park! ...,Positive,2023-01-15 12:30:00,User123,Twitter,#Nature #Park,15.0,30.0,USA,2023,1,15,12
1,1,1,Traffic was terrible this morning. ...,Negative,2023-01-15 08:45:00,CommuterX,Twitter,#Traffic #Morning,5.0,10.0,Canada,2023,1,15,8
2,2,2,Just finished an amazing workout! 💪 ...,Positive,2023-01-15 15:45:00,FitnessFan,Instagram,#Fitness #Workout,20.0,40.0,USA,2023,1,15,15
3,3,3,Excited about the upcoming weekend getaway! ...,Positive,2023-01-15 18:20:00,AdventureX,Facebook,#Travel #Adventure,8.0,15.0,UK,2023,1,15,18
4,4,4,Trying out a new recipe for dinner tonight. ...,Neutral,2023-01-15 19:55:00,ChefCook,Instagram,#Cooking #Food,12.0,25.0,Australia,2023,1,15,19
5,5,5,Feeling grateful for the little things in lif...,Positive,2023-01-16 09:10:00,GratitudeNow,Twitter,#Gratitude #PositiveVibes,25.0,50.0,India,2023,1,16,9
6,6,6,Rainy days call for cozy blankets and hot coc...,Positive,2023-01-16 14:45:00,RainyDays,Facebook,#RainyDays #Cozy,10.0,20.0,Canada,2023,1,16,14
7,7,7,The new movie release is a must-watch! ...,Positive,2023-01-16 19:30:00,MovieBuff,Instagram,#MovieNight #MustWatch,15.0,30.0,USA,2023,1,16,19
8,8,8,Political discussions heating up on the timel...,Negative,2023-01-17 08:00:00,DebateTalk,Twitter,#Politics #Debate,30.0,60.0,USA,2023,1,17,8
9,9,9,Missing summer vibes and beach days. ...,Neutral,2023-01-17 12:20:00,BeachLover,Facebook,#Summer #BeachDays,18.0,35.0,Australia,2023,1,17,12


In [80]:
df.shape

(732, 15)

## dropping unnecessary columns

In [82]:
df = df[["Text", "Sentiment"]].dropna()

print(df.head())
print(df["Sentiment"].value_counts())

                                                Text Sentiment
0   Enjoying a beautiful day at the park!        ...  positive
1   Traffic was terrible this morning.           ...  negative
2   Just finished an amazing workout! 💪          ...  positive
3   Excited about the upcoming weekend getaway!  ...  positive
5   Feeling grateful for the little things in lif...  positive
Sentiment
positive    45
negative     4
Name: count, dtype: int64


In [83]:
# Ondoa missing values
df = df.dropna(subset=["Text", "Sentiment"]).copy()

# Safisha labels
df["Sentiment"] = df["Sentiment"].astype(str).str.strip().str.lower()

# Baki na positive na negative
df = df[df["Sentiment"].isin(["positive", "negative"])].copy()

print(df["Sentiment"].value_counts())
print("Dataset shape:", df.shape)

Sentiment
positive    45
negative     4
Name: count, dtype: int64
Dataset shape: (49, 2)


In [84]:
print(df.columns.tolist())

['Text', 'Sentiment']


In [85]:
df.head(2)

,Text,Sentiment
0,Enjoying a beautiful day at the park! ...,positive
1,Traffic was terrible this morning. ...,negative


## performing train_test and split

In [87]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["Sentiment"]
)

print("Train:", len(train_df))
print("Test:", len(test_df))

Train: 39
Test: 10


In [90]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_df[["Text", "Sentiment"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["Text", "Sentiment"]],
    preserve_index=False
)

print(train_dataset)
print(test_dataset)

Dataset({
    features: ['Text', 'Sentiment'],
    num_rows: 39
})
Dataset({
    features: ['Text', 'Sentiment'],
    num_rows: 10
})


In [91]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)